# Notebook 5: Model Evaluation & Comparison
Side-by-side SARIMAX vs Prophet metrics, best model selection, and business insights.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

arima_m   = pd.read_csv('../data/processed/arima_metrics.csv')
prophet_m = pd.read_csv('../data/processed/prophet_metrics.csv')

arima_m.columns   = ['Product', 'ARIMA_MAE', 'ARIMA_RMSE', 'ARIMA_MAPE']
prophet_m.columns = ['Product', 'Prophet_MAE', 'Prophet_RMSE', 'Prophet_MAPE']

combined = arima_m.merge(prophet_m, on='Product')
combined['Best_Model'] = combined.apply(
    lambda r: 'SARIMAX' if r['ARIMA_MAPE'] < r['Prophet_MAPE'] else 'Prophet', axis=1
)
print(combined.to_string(index=False))

## 1. MAPE Comparison — SARIMAX vs Prophet

In [ ]:
x = np.arange(len(combined))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 6))
bars1 = ax.bar(x - width/2, combined['ARIMA_MAPE'], width, label='SARIMAX', color='steelblue', alpha=0.85)
bars2 = ax.bar(x + width/2, combined['Prophet_MAPE'], width, label='Prophet', color='coral', alpha=0.85)

ax.set_xlabel('Product')
ax.set_ylabel('MAPE (%)')
ax.set_title('Model Comparison: MAPE (%) by Product — Lower is Better', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(combined['Product'], rotation=30, ha='right')
ax.legend()
ax.axhline(10, color='gray', linestyle='--', linewidth=1, label='10% threshold')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('../reports/figures/14_mape_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. RMSE Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - width/2, combined['ARIMA_RMSE'], width, label='SARIMAX', color='steelblue', alpha=0.85)
ax.bar(x + width/2, combined['Prophet_RMSE'], width, label='Prophet', color='coral', alpha=0.85)
ax.set_title('RMSE Comparison by Product', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(combined['Product'], rotation=30, ha='right')
ax.set_ylabel('RMSE (Units)')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/15_rmse_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Best Model per Product

In [ ]:
print('=== Best Model per Product (by MAPE) ===')
for _, row in combined.iterrows():
    winner = row['Best_Model']
    mape_w = row['ARIMA_MAPE'] if winner == 'SARIMAX' else row['Prophet_MAPE']
    print(f"{row['Product']:<22} → {winner:<10} (MAPE: {mape_w:.2f}%)")

print(f"\nSARIMAX wins: {(combined['Best_Model']=='SARIMAX').sum()} / {len(combined)} products")
print(f"Prophet wins: {(combined['Best_Model']=='Prophet').sum()} / {len(combined)} products")

## 4. Business Insights from Forecasting

In [ ]:
df_agg = pd.read_csv('../data/processed/featured_aggregated_data.csv', parse_dates=['date'])

# Insight 1: Which products need highest safety stock?
volatility = df_agg.groupby('product_name')['total_units_sold'].std() / df_agg.groupby('product_name')['total_units_sold'].mean() * 100
volatility = volatility.sort_values(ascending=False).rename('CoV_%')

print('=== Demand Volatility (Coefficient of Variation) ===')
print('Higher CoV → higher safety stock needed')
print(volatility.round(2).to_string())

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(volatility.index, volatility.values, color='steelblue', alpha=0.8)
ax.set_title('Demand Volatility by Product (Coefficient of Variation %)', fontweight='bold')
ax.set_ylabel('CoV (%)')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('../reports/figures/16_demand_volatility.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Final summary table for report
summary = combined[['Product', 'ARIMA_MAPE', 'Prophet_MAPE', 'Best_Model']].copy()
summary.columns = ['Product', 'SARIMAX MAPE (%)', 'Prophet MAPE (%)', 'Best Model']
summary.to_csv('../data/processed/final_model_comparison.csv', index=False)

print('=== FINAL SUMMARY TABLE ===')
print(summary.to_string(index=False))
print('\nAll evaluation figures saved. Project modelling complete!')